In [1]:
import pandas as pd
import numpy as np

customer = pd.read_csv("train/train_customer_info.csv")
finance = pd.read_csv("train/train_finance_profile.csv")
transaction = pd.read_csv("train/train_transaction_history.csv")
target = pd.read_csv("train/train_targets.csv")

In [2]:
transaction["trans_date"] = pd.to_datetime(transaction["trans_date"])

In [3]:
trans_agg = transaction.groupby("customer_id").agg({
    "trans_amount": ["sum", "mean", "max", "min", "count"],
    "is_installment": "mean"
})

In [4]:
trans_agg.columns = [
    "total_amount",
    "avg_amount",
    "max_amount",
    "min_amount",
    "purchase_count",
    "installment_ratio"
]

trans_agg = trans_agg.reset_index()

In [5]:
recent_trans = transaction.groupby("customer_id")["trans_date"].max().reset_index()
recent_trans.columns = ["customer_id", "last_purchase_date"]

In [6]:
category_div = transaction.groupby("customer_id")["item_category"].nunique().reset_index()
category_div.columns = ["customer_id", "category_diversity"]

In [7]:
latest_date = transaction["trans_date"].max()
last_30 = transaction[
    transaction["trans_date"] >= latest_date - pd.Timedelta(days=30)
]

recent_30 = last_30.groupby("customer_id")["trans_amount"].sum().reset_index()
recent_30.columns = ["customer_id", "recent_30d_amount"]

In [8]:
transaction_features = trans_agg \
    .merge(recent_trans, on="customer_id", how="left") \
    .merge(category_div, on="customer_id", how="left") \
    .merge(recent_30, on="customer_id", how="left")

In [9]:
df = customer.merge(finance, on="customer_id", how="left")
df = df.merge(transaction_features, on="customer_id", how="left")
df = df.merge(target, on="customer_id", how="left")

In [10]:
df.fillna(0, inplace=True)

In [11]:
print(df.shape)
df.head()

(60000, 27)


,customer_id,join_date,age,gender,region_code,is_married,prefer_category,income_group,credit_score,num_active_cards,...,avg_amount,max_amount,min_amount,purchase_count,installment_ratio,last_purchase_date,category_diversity,recent_30d_amount,target_churn,target_ltv
0,C000001,2020-04-12,36,F,R03,1,Grocery,G4,713,6,...,51784.571429,227979,12136,21,0.238095,2023-12-27,5,160129.0,0,556691.00
1,C000002,2021-03-11,32,F,R02,0,Home,G3,869,5,...,65868.142857,266110,12019,14,0.000000,2023-12-29,4,182867.0,0,1460203.00
2,C000003,2022-05-10,41,M,R03,1,Fashion,G3,588,2,...,50788.625000,166157,13903,16,0.375000,2023-12-31,5,135028.0,0,605476.00
3,C000004,2020-09-27,23,F,R02,1,Electronics,G3,742,3,...,56230.000000,242701,15312,22,0.136364,2023-12-24,5,125901.0,0,1034150.00
4,C000006,2020-03-12,31,F,R04,0,Grocery,G3,611,5,...,73860.214286,227673,29072,14,0.285714,2023-11-25,5,0.0,1,76083.15
